In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm


%matplotlib inline

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
print(f"Dataset downloaded to: {path}")
print(f"CSV file loaded from: {csv_path}")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
print("DataFrame Info:")
df.info()

In [ ]:
# Task 4: Write your code here:
print("\nDataFrame Description:")
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Delivery_Time')
plt.title('Distribution of Target Variable (Delivery_Time)')
plt.xlabel('Delivery_Time')
plt.ylabel('count')
plt.show()


In [ ]:
# Task 1: Write your code here:
clean_df = df
clean_df = clean_df.drop('Order_ID', axis=1)

clean_df.head()

In [ ]:
# Task 2: Write your code here:
# Analyze missing values
missing_percentage = (clean_df.isnull().sum() / len(clean_df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

print(f"Before: {clean_df.shape}")
clean_df = clean_df.dropna()
print(f"After: {clean_df.shape}")


In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = clean_df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  clean_df[col] = le.fit_transform(clean_df[col])
  label_encoders[col] = le
clean_df

In [ ]:
# Task 5: Write your code here:
numerical_cols = clean_df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
clean_df[numerical_cols] = scaler.fit_transform(clean_df[numerical_cols])
clean_df.head()


In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(clean_df, "Delivery_Time")
#Target looks imbalanced

In [ ]:
from sklearn.model_selection import train_test_split
# Task 1: Write your code here:
print("Separating target variable and features...")
X = clean_df.drop('Delivery_Time', axis=1)
y = clean_df['Delivery_Time']

print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Applying One-Hot Encoding to feature DataFrame 'X'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))

print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")
print(f"Shape of y_encoded: {y_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])

print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.3, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier



# Mean Squared Error in NumPy
def mean_squared_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum((y_hat - y) ** 2)


def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_squared_error(y, y_hat)
    losses.append(loss)

  return theta, losses



num_classes = len(np.unique(y_encoded))
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = categorical_cross_entropy(y_val_one_hot, y_pred_proba)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)


In [ ]:
# Task 1: Write your code here:
average_losses = np.mean(lr_losses, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: